# Create ANR Work Funders (funder-reported grant -> work linkages)

ANR (Agence Nationale de la Recherche, `F4320320883`) sends us a bulk file mapping their
grants to the DOIs of the publications they funded, compiled from researchers' end-of-project
reporting plus HAL / OpenAlex / WOS. This is the **trusted funder-reported** model (same as
`CreateNWOWorkAwards`): we assert the grant->work edge even when none of our own pipelines
derive it independently.

**Upstream:** `scripts/local/anr_work_links_to_s3.py` ->
`s3://openalex-ingest/awards/anr/anr_work_links.parquet`

**Output:** `openalex.awards.anr_work_funders (work_id, funder_id, award_ids[])`,
consumed by `WorkAwards.ipynb` at **priority 3** (funder-reported tier, peer of NWO/GTR).

### Prerequisite
Run **`CreateANRAwards`** first. The grant->work join keys on
`anr_awards.funder_award_id` (= `projet.code_decision`, e.g. `ANR-12-CHRI-0003`). As of
2026-06-19 the deployed `anr_awards` was stale (13.4K rows) and matched only 34% of the
linkage grants; a refresh from the current S3 parquet (34.4K rows) lifts grant coverage to
**99.2%**. Do not run this notebook against a stale `anr_awards`.

### Resolution strategy
Resolve each DOI against `openalex_works.doi` (canonical). ANR also ships its own
`ID Open Alex short` W-id, but on a 3K sample ~1.4% of those W-ids disagree with the DOI
(stale/merged), and the W-id never resolves a work the DOI doesn't -- so we ignore it and
re-resolve from the DOI. ~99.8% of DOIs resolve.

In [ ]:
%sql
-- Staging table from the S3 parquet produced by anr_work_links_to_s3.py
CREATE OR REPLACE TABLE openalex.awards.anr_work_links_raw
USING delta
AS
SELECT
    *,
    current_timestamp() AS databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/anr/anr_work_links.parquet`;

In [ ]:
%sql
-- Sanity: row count + key population (expect ~268.9K rows, ~2.1K with no grant)
SELECT
    COUNT(*)                                            AS total_rows,
    COUNT(doi)                                          AS has_doi,
    SUM(CASE WHEN SIZE(anr_grants) > 0 THEN 1 ELSE 0 END) AS has_grant,
    SUM(CASE WHEN SIZE(anr_grants) > 1 THEN 1 ELSE 0 END) AS multi_grant,
    SUM(CASE WHEN internal_source THEN 1 ELSE 0 END)    AS from_final_reports
FROM openalex.awards.anr_work_links_raw;

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.anr_work_funders
USING delta
AS
WITH
-- 1) Explode '|'-multi-grant rows into one (doi, grant) pair each.
exploded AS (
  SELECT
    lower(r.doi)                          AS doi,
    TRIM(g.grant_id)                      AS funder_award_id
  FROM openalex.awards.anr_work_links_raw r
  LATERAL VIEW EXPLODE(r.anr_grants) g AS grant_id
  WHERE r.doi IS NOT NULL
    AND g.grant_id IS NOT NULL
    AND TRIM(g.grant_id) <> ''
),
-- 2) Resolve DOI -> OpenAlex work (canonical; handles merges that stale W-ids miss).
doi_resolved AS (
  SELECT DISTINCT
    w.id           AS work_id,
    e.funder_award_id
  FROM exploded e
  JOIN openalex.works.openalex_works w
    ON lower(w.doi) = CONCAT('https://doi.org/', e.doi)
),
-- 3) Confirm the award entity exists (must run CreateANRAwards first) and pick up funder_id.
--    Exact join on funder_award_id (= projet.code_decision). Grants not in our ingest are
--    dropped: ~0.8% of the file, almost all pre-2007 programs absent from data.gouv.fr.
with_award AS (
  SELECT
    d.work_id,
    a.funder_id,
    a.funder_award_id
  FROM doi_resolved d
  JOIN openalex.awards.anr_awards a
    ON a.funder_award_id = d.funder_award_id
  WHERE d.work_id IS NOT NULL
)
SELECT
  work_id,
  funder_id,
  ARRAY_DISTINCT(COLLECT_LIST(funder_award_id)) AS award_ids
FROM with_award
GROUP BY work_id, funder_id;

## Validation

In [ ]:
%sql
-- Row counts, key uniqueness, award fill
SELECT
  COUNT(*)                                          AS total_rows,
  COUNT(DISTINCT CONCAT(work_id, ':', funder_id))   AS distinct_keys,
  COUNT(DISTINCT work_id)                           AS distinct_works,
  COUNT(DISTINCT funder_id)                         AS distinct_funders,
  SUM(SIZE(award_ids))                              AS total_edges,
  COUNT(CASE WHEN SIZE(award_ids) > 1 THEN 1 END)   AS works_multi_award
FROM openalex.awards.anr_work_funders;

In [ ]:
%sql
-- Composite-key uniqueness (expect 0) and grant-level coverage vs the linkage population
WITH dup AS (
  SELECT work_id, funder_id, COUNT(*) AS n
  FROM openalex.awards.anr_work_funders
  GROUP BY work_id, funder_id HAVING n > 1
),
link_grants AS (  -- distinct grants asserted in the source file
  SELECT COUNT(DISTINCT TRIM(g.grant_id)) AS n
  FROM openalex.awards.anr_work_links_raw r
  LATERAL VIEW EXPLODE(r.anr_grants) g AS grant_id
  WHERE TRIM(g.grant_id) <> ''
),
resolved_grants AS (  -- distinct grants that produced at least one edge
  SELECT COUNT(DISTINCT a.funder_award_id) AS n
  FROM (SELECT EXPLODE(award_ids) AS award_id FROM openalex.awards.anr_work_funders) t
  JOIN openalex.awards.anr_awards a ON a.funder_award_id = t.award_id
)
SELECT
  (SELECT COUNT(*) FROM dup)                                                 AS duplicate_keys,
  (SELECT n FROM link_grants)                                               AS grants_in_file,
  (SELECT n FROM resolved_grants)                                           AS grants_with_a_linked_work,
  ROUND(100.0 * (SELECT n FROM resolved_grants) / (SELECT n FROM link_grants), 1) AS pct_grants_covered;

In [ ]:
%sql
-- DOI -> work resolution rate: distinct source DOIs vs distinct DOIs that resolved.
WITH src AS (
  SELECT COUNT(DISTINCT lower(doi)) AS n
  FROM openalex.awards.anr_work_links_raw WHERE doi IS NOT NULL
),
res AS (
  SELECT COUNT(DISTINCT lower(r.doi)) AS n
  FROM openalex.awards.anr_work_links_raw r
  JOIN openalex.works.openalex_works w
    ON lower(w.doi) = CONCAT('https://doi.org/', lower(r.doi))
)
SELECT (SELECT n FROM src) AS distinct_dois,
       (SELECT n FROM res) AS dois_resolved_to_work,
       ROUND(100.0 * (SELECT n FROM res) / (SELECT n FROM src), 1) AS pct_dois_resolved;

In [ ]:
%sql
-- Spot-check: a few resolved edges with their grant + work
SELECT wf.work_id, wf.funder_id, wf.award_ids, w.doi, w.title
FROM openalex.awards.anr_work_funders wf
JOIN openalex.works.openalex_works w ON w.id = wf.work_id
LIMIT 20;